## Load EC3 EPD Data

### Load dataframe

In [1]:
import pandas as pd

# Load in another notebook
df = pd.read_pickle('../02_processed_data/epd_data_cleaned.pkl')


In [2]:
df.head()

,gwp,lightweight,id,gwp_per_category_declared_unit,concrete_compressive_strength_28d,name,updated_on,open_xpd_uuid,created_on,uncertainty_factor,...,plant_or_group.name,plant_or_group.updated_on,plant_or_group.created_on,plant_or_group.type,cementitious.ggbs,concrete_aggregate_size_max,cementitious.fly_ash,gwp_val_per_cy,created_date_formatted,Compressive_Strength
0,319 kgCO2e,False,a213fdb0dbeb449ebb9595c6eba75af4,319 kgCO2e,55.2 MPa,Mix SRM80S5BSNA,2024-05-24T15:40:10.256074Z,ec3wdhqk,2024-04-26T21:02:50.898997Z,1.094574,...,Long Island City,2024-08-29T15:11:43.148443Z,2023-05-03T08:37:37.808959Z,Plant,0.50,NaN,NaN,243.893045,2024-04-26 21:02:50.898997+00:00,8000
1,318 kgCO2e,False,c773e40a243741a4a13c43af17a42fa9,318 kgCO2e,31 MPa,Mix 390863,2024-05-24T16:03:40.536347Z,ec3kkxe7,2024-04-26T20:25:15.276652Z,1.094574,...,Hilroy,2024-07-19T16:00:02.097740Z,2024-02-20T22:54:03.331812Z,Plant,0.40,0.75 in,NaN,243.128490,2024-04-26 20:25:15.276652+00:00,4500
2,318 kgCO2e,False,0f36072df02344ca8eb9f215a8954a69,318 kgCO2e,34.5 MPa,Mix 390866,2024-05-24T15:17:11.241268Z,ec36963f,2024-04-26T20:23:56.162527Z,1.094574,...,Hilroy,2024-07-19T16:00:02.097740Z,2024-02-20T22:54:03.331812Z,Plant,0.50,0.75 in,NaN,243.128490,2024-04-26 20:23:56.162527+00:00,5000
3,291 kgCO2e,False,3a6aa86f423a469f981115b68838be3c,291 kgCO2e,37.9 MPa,Mix 506-S45,2024-05-24T15:31:40.346516Z,ec3ndbwz,2024-04-26T20:12:49.431963Z,1.094574,...,Prestress – Spokane Valley (Washington),2024-08-23T18:52:22.419903Z,2023-10-18T11:47:14.114325Z,Plant,0.45,NaN,NaN,222.485505,2024-04-26 20:12:49.431963+00:00,5500
4,289 kgCO2e,False,abb548d9550a450492870f5084dd4325,289 kgCO2e,34.5 MPa,Mix 50IM40%SL,2024-05-24T15:00:31.466560Z,ec3ptdhb,2024-04-26T20:02:37.949597Z,1.094574,...,Bronx,2024-07-23T07:58:43.043312Z,2023-05-04T20:27:47.059512Z,Plant,0.40,0.75 in,NaN,220.956395,2024-04-26 20:02:37.949597+00:00,5000


In [7]:
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd

# Filter data: only include Compressive_Strength >= 2500 and values with 20+ samples
df_filtered = df[df['Compressive_Strength'] >= 2500].copy()
strength_counts = df_filtered['Compressive_Strength'].value_counts()
valid_strengths = strength_counts[strength_counts >= 20].index
df_filtered = df_filtered[df_filtered['Compressive_Strength'].isin(valid_strengths)].copy()

# Create a categorical variable for SCM type
def classify_scm(row):
    has_fly_ash = pd.notna(row.get('cementitious.fly_ash')) and row.get('cementitious.fly_ash') > 0
    has_ggbs = pd.notna(row.get('cementitious.ggbs')) and row.get('cementitious.ggbs') > 0

    if has_fly_ash and has_ggbs:
        return 'Both Fly Ash & Slag'
    elif has_fly_ash:
        return 'Fly Ash'
    elif has_ggbs:
        return 'Slag'
    else:
        return 'Neither'

df_filtered['scm_type'] = df_filtered.apply(classify_scm, axis=1)

# Round gwp values for cleaner display
df_filtered['gwp_rounded'] = df_filtered['gwp_val_per_cy'].round(0)

# Sort compressive strengths and get counts
strength_order = sorted(df_filtered['Compressive_Strength'].unique())
strength_counts_dict = df_filtered['Compressive_Strength'].value_counts().to_dict()

# Define a nice color palette for SCM types
color_map = {
    'Fly Ash': '#FF6B6B',              # Coral red
    'Slag': '#4ECDC4',                 # Turquoise
    'Both Fly Ash & Slag': '#FFE66D',  # Yellow
    #'Neither': '#95A5A6'               # Gray
}

# Create the box plot grouped by Compressive_Strength, colored by SCM type
fig = px.box(
    df_filtered,
    x='Compressive_Strength',
    y='gwp_rounded',
    title='GWP Distribution by Compressive Strength and SCM Type',
    labels={
        'Compressive_Strength': 'Compressive Strength',
        'gwp_rounded': 'GWP (kg CO₂e per cubic yard)',
        'scm_type': 'SCM Type'
    },
    category_orders={'Compressive_Strength': strength_order},
    hover_data=['scm_type', 'name'],
    color='scm_type',
    color_discrete_map=color_map
)

# Add all data points as dots with transparency and jitter
fig.update_traces(
    boxpoints='all',
    jitter=0.3,
    pointpos=0,
    marker=dict(size=5, opacity=0.6, line=dict(width=0.5, color='white'))
)

# Add vertical lines between compressive strength buckets
y_min = df_filtered['gwp_rounded'].min()
y_max = df_filtered['gwp_rounded'].max()

for i in range(len(strength_order) - 1):
    # Position line between current and next strength value
    x_position = (strength_order[i] + strength_order[i+1]) / 2

    fig.add_shape(
        type="line",
        x0=x_position,
        y0=y_min - 20,  # Extend slightly below
        x1=x_position,
        y1=y_max + 20,  # Extend slightly above
        line=dict(
            color="rgba(128, 128, 128, 0.3)",
            width=1,
            dash="dash"
        ),
        layer="below"
    )

# Update y-axis
fig.update_yaxes(
    title_text='GWP (kg CO₂e per cubic yard)',
    showgrid=True,
    gridcolor='lightgrey'
)

# Update x-axis to include counts in labels
x_labels_with_counts = [f"{int(strength)} psi [{strength_counts_dict[strength]}]"
                        for strength in strength_order]

fig.update_xaxes(
    ticktext=x_labels_with_counts,
    tickvals=strength_order,
    tickangle=-45
)

# Update layout
fig.update_layout(
    title={
        'text': 'GWP Distribution by Compressive Strength and SCM Type',
        'x': 0.5,
        'xanchor': 'center',
        'yanchor': 'top',
        'font': {'size': 16, 'family': 'Arial', 'color': 'black'}
    },
    plot_bgcolor='white',
    paper_bgcolor='white',
    height=700,
    width=1200,
    legend={
        'title': 'SCM Type',
        'orientation': 'v',
        'yanchor': 'top',
        'y': 0.99,
        'xanchor': 'left',
        'x': 1.01
    }
)

# Show summary statistics
print(f"Total samples: {len(df_filtered)}")
print(f"\nSamples by Compressive Strength:")
for strength in strength_order:
    count = strength_counts_dict[strength]
    print(f"  {int(strength)} psi: {count} samples")
print(f"\nSamples by SCM Type:")
print(df_filtered['scm_type'].value_counts())

# Show the plot
fig.show()

Total samples: 4757

Samples by Compressive Strength:
  2500 psi: 66 samples
  3000 psi: 612 samples
  3500 psi: 329 samples
  4000 psi: 1435 samples
  4500 psi: 509 samples
  5000 psi: 820 samples
  5500 psi: 110 samples
  6000 psi: 623 samples
  6500 psi: 20 samples
  7000 psi: 105 samples
  8000 psi: 105 samples
  10000 psi: 23 samples

Samples by SCM Type:
scm_type
Fly Ash                3816
Slag                    850
Both Fly Ash & Slag      91
Name: count, dtype: int64


### Save Plot

In [9]:
# Save the plot as HTML
import os
output_path = '../05_outputs/gwp_by_compressive_strength_scm.html'
fig.write_html(output_path)